In [1]:
# %% [markdown]
# Notebook ETL – leitura em batch, validações e write-back

In [2]:
import os, sys, importlib

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
logs_dir = os.path.join(project_root, "logs")
if logs_dir not in sys.path:
    sys.path.insert(0, logs_dir)
import logging_setup

importlib.reload(logging_setup)
from logging_setup import get_logger  # importa e já faz setup_logging

logger = get_logger(__name__)

In [3]:
# %% [code]
import os
import sys
from dotenv import load_dotenv

project_root = (
    os.getcwd()
)  # supondo que o notebook esteja em /home/debrito/Documentos/etl_debrito
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ── Carrega variáveis de ambiente de .env (opcional) ───────────────────────
load_dotenv()

False

In [4]:
# 2 %% [code]
import math
import numpy as np
from typing import Any


def _to_json_safe(x: Any) -> Any:
    """
    Internal helper to convert arbitrary objects into JSON-safe primitives.
    """
    if x is None:
        return None
    if isinstance(x, (int, str, bool)):
        return x
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if (math.isnan(x) or math.isinf(x)) else float(x)
    if isinstance(x, (list, tuple, set)):
        return [_to_json_safe(item) for item in x]
    if isinstance(x, dict):
        return {k: _to_json_safe(v) for k, v in x.items()}
    # Fallback: stringify anything else
    return str(x)


def json_safe(obj: Any) -> Any:
    """
    Recursively converts `obj` into structures 100% serializable to JSON.
    """
    return _to_json_safe(obj)

In [ ]:
# 3 %% [code]
# Flags de gravação (ajuste conforme necessidade)
WRITE_BACK_ORIGIN = True  # grava na aba-origem (meta*, tiktok*, …)
WRITE_BACK_DEST = True  # grava nas abas-modelo (modelo*)
DRY_RUN_DEST = False  # True = simula write-back destino

# Credenciais e identificador da planilha
CREDS_PATH = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
SPREADSHEET_ID = os.getenv(
    "GOOGLE_SHEET_ID", "1jPFLqg7HIDZoxCwQacMpCS_bCKfRsnisvEniiHLljiE"
)

# Abas de origem a processar, agrupadas por plataforma
SHEET_NAMES = [
    # Meta (Facebook / Instagram)
    "metaGeral",
    "metaIdade",
    "metaGenero",
    "metaRegiao",
    "metaAlcance",
    # TikTok
    "tiktokGeral",
    "tiktokIdade",
    "tiktokGenero",
    "tiktokRegiao",
    "tiktokAlcance",
    # Pinterest
    "pinterestGeral",
    "pinterestGenero",
    "pinterestIdade",
    "pinterestRegiao",
    "pinterestAlcance",
    # LinkedIn
    "linkedinGeral",
    "linkedinRegiao",
    "linkedinAlcance",
    # Google Analytics
    "GAGeral",
]

In [ ]:
# 4
# %% [code]
import os
import importlib

# módulos principais e helpers recarregados
import extract.sheets_fetcher as sf_mod
import treat.treat_pipeline as tp_mod
import treat.platforms as platforms_mod
import treat.platforms.linkedin as linkedin_mod
import treat.platforms.tiktok as tiktok_mod
import treat.platforms.pinterest as pinterest_mod
import treat.platforms.meta as meta_mod
import treat.platforms.ga as ga_mod
import load.origin_writer as ow_mod
import load.dest_writer as dw_mod
import treat.utils.renomeacoes as rn_mod
import treat.utils.preview_links as prev_mod
import treat.utils.atribuicoes_via_lookup as atrib_mod
import treat.utils.substitute_origin_values as sub_mod
import treat.utils.preprocess_utils as pre_mod
import treat.utils.geo_normalize as geo_mod

# hot-reload de todos os módulos alterados durante o desenvolvimento
for m in (
    sf_mod,
    tp_mod,
    platforms_mod,
    linkedin_mod,
    tiktok_mod,
    pinterest_mod,
    meta_mod,
    ga_mod,
    ow_mod,
    dw_mod,
    rn_mod,
    prev_mod,
    atrib_mod,
    sub_mod,
    pre_mod,
    geo_mod,
):
    importlib.reload(m)

In [ ]:
# %% [code]
# Cell 5: definição do helper run_etl_for_sheet
import pandas as pd
import json
from pprint import pp
from typing import Dict

from logs.logging_setup import get_logger

logger = get_logger(__name__)

from extract.sheets_fetcher import SheetsFetcher
from treat.treat_pipeline import TreatPipeline
from treat.utils.renomeacoes import (
    renomeacao_geral,
    renomear_colunas_origem_para_modelo,
)
from treat.utils.campos_calculados import calcular_engajamento_total, gerar_id
from load.origin_writer import write_back_origin
from load.dest_writer import write_back_for_sheet

# Instância única do fetcher (retry/backoff/cache interno)
fetcher = SheetsFetcher(
    spreadsheet_id=SPREADSHEET_ID,
    creds_path=CREDS_PATH,
)


def run_etl_for_sheet(
    *,
    sheet: str,
    wb_origin_flag: bool,
    wb_dest_flag: bool,
    dry_run_dest: bool,
    preloaded_raw: pd.DataFrame,
) -> Dict[str, pd.DataFrame | dict]:
    """
    Executa o fluxo completo para uma aba e devolve:
      { "dest": DataFrame destino (ou vazio), "taxo": relatório de taxonomia }
    """
    # 1) Dados brutos já carregados
    df_raw = preloaded_raw

    # 2) Tratamento via pipeline
    pipeline = TreatPipeline(
        creds_path=CREDS_PATH,
        spreadsheet_id=SPREADSHEET_ID,
        sheet_name=sheet,
        mapping_renomeacao=renomeacao_geral,
        write_back=wb_origin_flag,
    )
    df_ok = pipeline.run(df_raw)

    # 3) Relatório de taxonomia
    taxo_report = getattr(pipeline, "_last_taxo_report", {})
    pp(json.dumps(taxo_report, default=str), width=120)

    # 4) Write-back na aba de origem (apenas quando não for Pinterest demográfico)
    is_pinterest_dim = sheet.lower() in {
        "pinterestgenero",
        "pinterestidade",
        "pinterestregiao",
    }

    if not is_pinterest_dim:
        # grava correções de pré-processamento in-place
        _ = write_back_origin(
            df_raw=df_raw,
            df_ok=df_ok,
            creds_path=CREDS_PATH,
            spreadsheet_id=SPREADSHEET_ID,
            sheet_name=sheet,
            write_back=wb_origin_flag,
            dry_run=not wb_origin_flag,
        )
    else:
        logger.debug(
            "🔸 %s: pulando write-back de origem (já feito dentro de pipeline)", sheet
        )

    # 5) Preparar DataFrame de destino (modelo)
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 6) Write-back de destino  ──────────────────────────────────────────────
    if sheet.lower().startswith("ga"):
        logger.info("🔸 %s: write-back de destino ignorado (Google Analytics)", sheet)
        df_dest = pd.DataFrame()  # retorna DataFrame vazio
    else:
        df_dest = write_back_for_sheet(
            df_model,
            sheet_name=sheet,
            creds_path=CREDS_PATH,
            spreadsheet_id=SPREADSHEET_ID,
            write_back=wb_dest_flag,
            dry_run=dry_run_dest,
        )
        if df_dest is None:
            df_dest = pd.DataFrame()

    # 7) Retorno
    return {"dest": df_dest, "taxo": taxo_report}

In [ ]:
# %% [markdown]
# ## 🔧 Resolução de Problemas de Autenticação

# Se você receber **HttpError 403 "The caller does not have permission"**, verifique:

# ### 1. Credenciais (creds.json)
# - Arquivo `creds.json` existe e está no local correto
# - Credenciais não estão expiradas
# - Conta de serviço tem as permissões necessárias

# ### 2. Compartilhamento da Planilha
# - Planilha deve ser compartilhada com o email da conta de serviço
# - Email da conta de serviço está em `creds.json` → `client_email`
# - Permissão mínima: **Editor** (para write-back) ou **Viewer** (só leitura)

# ### 3. ID da Planilha
# - Verifique se `SPREADSHEET_ID` está correto
# - ID deve ser extraído da URL: `https://docs.google.com/spreadsheets/d/{ID}/edit`

# ### 4. Cotas da API
# - Google Sheets API tem limites de uso
# - Aguarde alguns minutos se as cotas foram excedidas

# ### 5. Verificação Rápida das Credenciais

In [ ]:
# %% [code]
# 🧪 TESTE DE CONECTIVIDADE - Verificar se a planilha foi compartilhada corretamente
import gspread
import google.auth
from googleapiclient.errors import HttpError

print("🧪 Testando conectividade com a planilha...")
print(f"📧 Conta de serviço: diginostic@diginostic.iam.gserviceaccount.com")
print(f"🔗 Planilha ID: {SPREADSHEET_ID}")

try:
    # Teste básico de conectividade
    creds, _ = google.auth.load_credentials_from_file(
        CREDS_PATH, scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
    )
    gc = gspread.authorize(creds)
    
    # Tentar abrir a planilha
    ss = gc.open_by_key(SPREADSHEET_ID)
    print(f"✅ Planilha acessada com sucesso!")
    print(f"📝 Nome da planilha: {ss.title}")
    
    # Listar as primeiras abas para confirmar acesso
    worksheets = ss.worksheets()[:5]  # Apenas as primeiras 5 abas
    print(f"📋 Primeiras abas encontradas:")
    for i, ws in enumerate(worksheets, 1):
        print(f"  {i}. {ws.title} ({ws.row_count}×{ws.col_count})")
    
    # Teste de leitura em uma aba pequena
    try:
        test_sheet = worksheets[0]
        # Tentar ler apenas uma célula para testar
        cell_value = test_sheet.acell('A1').value
        print(f"✅ Teste de leitura bem-sucedido! Célula A1 da aba '{test_sheet.title}': '{cell_value}'")
        
        print("\n🎉 SUCESSO! A planilha está compartilhada corretamente!")
        print("✨ Você pode executar o pipeline principal agora.")
        
    except Exception as read_error:
        print(f"⚠️ Planilha acessada, mas erro na leitura: {read_error}")
    
except HttpError as e:
    if e.resp.status == 403:
        print("❌ ERRO: Planilha não compartilhada ou permissões insuficientes")
        print("\n📋 PASSOS PARA CORRIGIR:")
        print("1. Abra a planilha no Google Sheets:")
        print("   https://docs.google.com/spreadsheets/d/1jPFLqg7HIDZoxCwQacMpCS_bCKfRsnisvEniiHLljiE/edit")
        print("2. Clique em 'Compartilhar' (canto superior direito)")
        print("3. Adicione este email: diginostic@diginostic.iam.gserviceaccount.com")
        print("4. Selecione 'Editor' como permissão")
        print("5. Clique em 'Enviar'")
        print("6. Execute esta célula novamente")
    elif e.resp.status == 404:
        print("❌ ERRO: Planilha não encontrada (ID incorreto)")
        print("Verifique se o SPREADSHEET_ID está correto")
    else:
        print(f"❌ ERRO HTTP {e.resp.status}: {e}")
        
except Exception as e:
    print(f"❌ ERRO inesperado: {e}")
    print("Verifique suas credenciais e conexão com a internet")

In [ ]:
# %% [code]
# Verificação das credenciais e configurações
import json
import os
from pathlib import Path

print("🔍 Verificando configurações...")
print(f"CREDS_PATH: {CREDS_PATH}")
print(f"SPREADSHEET_ID: {SPREADSHEET_ID}")

# Verificar se arquivo de credenciais existe
creds_file = Path(CREDS_PATH)
if creds_file.exists():
    print("✅ Arquivo de credenciais encontrado")
    
    # Ler e mostrar informações básicas
    try:
        with open(CREDS_PATH, 'r') as f:
            creds_data = json.load(f)
        
        print(f"📧 Email da conta de serviço: {creds_data.get('client_email', 'N/A')}")
        print(f"🔑 Project ID: {creds_data.get('project_id', 'N/A')}")
        print(f"📝 Tipo: {creds_data.get('type', 'N/A')}")
        
    except Exception as e:
        print(f"❌ Erro ao ler credenciais: {e}")
else:
    print(f"❌ Arquivo de credenciais não encontrado em: {CREDS_PATH}")
    print("💡 Certifique-se de que o arquivo creds.json está no local correto")

print("\n📋 Próximos passos se houver erro 403:")
print("1. Compartilhe a planilha com o email da conta de serviço mostrado acima")
print("2. Dê permissão de 'Editor' para write-back ou 'Viewer' para apenas leitura")
print("3. Verifique se o SPREADSHEET_ID está correto")
print("4. Teste novamente")

In [ ]:
# %% [code]
%xmode verbose
import gc
import pandas as pd
from tqdm.auto import tqdm
from googleapiclient.errors import HttpError

from logs.logging_setup import get_logger
from load.dest_writer import prefetch_meta

logger = get_logger(__name__)


def process_sheets(
    fetcher,
    sheet_names: list[str],
    spreadsheet_id: str,
    write_origin: bool,
    write_dest: bool,
    dry_run: bool,
) -> dict[str, dict[str, object]]:
    """
    Lê todas as abas, faz prefetch de metadados e executa o ETL em cada aba.
    Retorna um dict com os resultados por aba.
    """
    logger.info("🔄 Iniciando processamento de abas")

    try:
        # 1) Leitura batch das abas
        raw_map = fetcher.get(sheet_names)
    except HttpError as e:
        if e.resp.status == 403:
            logger.error("❌ Erro de permissão do Google Sheets API (403)")
            logger.error("Possíveis causas:")
            logger.error("  • Credenciais inválidas ou expiradas")
            logger.error("  • Planilha não compartilhada com a conta de serviço")
            logger.error("  • ID da planilha incorreto")
            logger.error("  • Cotas da API excedidas")
            logger.error(f"Detalhes: {e}")
            return {}
        else:
            logger.error(f"❌ Erro HTTP {e.resp.status}: {e}")
            raise
    except Exception as e:
        logger.error(f"❌ Erro inesperado ao buscar dados: {e}")
        raise

    # 2) Copiando para evitar mutação in-place e validando tipos
    all_raw: dict[str, pd.DataFrame] = {}
    for name, df in raw_map.items():
        if isinstance(df, pd.DataFrame):
            all_raw[name] = df.copy()
        else:
            logger.error(
                f"Aba '{name}' não é um DataFrame (tipo={type(df)}); será ignorada."
            )

    # 3) Debug das colunas originais
    for name, df in all_raw.items():
        logger.debug(f"Aba '{name}' colunas originais: {df.columns.tolist()}")

    # 4) Prefetch de headers e IDs das abas-modelo
    try:
        prefetch_meta(fetcher, spreadsheet_id)
        logger.info("📥 Prefetch meta concluído – iniciando ETL por aba")
    except HttpError as e:
        logger.warning(f"⚠️ Erro no prefetch de metadados: {e}")
        logger.info("Continuando sem prefetch...")

    # 5) Processamento aba a aba
    results: dict[str, dict[str, object]] = {}
    for sheet in tqdm(sheet_names, desc="Processando abas"):
        try:
            df_raw = all_raw.get(sheet)
            if df_raw is None:
                logger.warning(f"Aba '{sheet}' não carregada; pulando ETL.")
                continue

            # Log específico para GA
            if sheet.lower().startswith("ga"):
                logger.info(
                    f"🔸 {sheet}: apenas write-back de origem; destino será ignorado"
                )

            out = run_etl_for_sheet(
                sheet=sheet,
                wb_origin_flag=write_origin,
                wb_dest_flag=write_dest,
                dry_run_dest=dry_run,
                preloaded_raw=df_raw,
            )

            results[sheet] = {"dest": out.get("dest"), "taxo": out.get("taxo")}
            logger.debug(f"Aba '{sheet}' processada com sucesso")

        except Exception as e:
            logger.exception(f"Erro ao processar aba '{sheet}': {e}")

        finally:
            # Garantir liberação de memória mesmo em caso de erro
            gc.collect()

    logger.info("✅ Processamento de todas as abas concluído")
    return results


# Chamando a função com tratamento de erro
try:
    results = process_sheets(
        fetcher=fetcher,
        sheet_names=SHEET_NAMES,
        spreadsheet_id=SPREADSHEET_ID,
        write_origin=WRITE_BACK_ORIGIN,
        write_dest=WRITE_BACK_DEST,
        dry_run=DRY_RUN_DEST,
    )
    if results:
        logger.info(f"✅ Processamento concluído. {len(results)} abas processadas.")
    else:
        logger.warning("⚠️ Nenhuma aba foi processada devido a erros.")
except Exception as e:
    logger.error(f"❌ Falha crítica no processamento: {e}")
    results = {}

In [ ]:
# %% [code]
# Cell 7: Validação de consistência de datas entre modelos e estatísticas de uso
from treat.treat_pipeline import BIParamLookup
from treat.utils.validations import validate_consistent_dates_across_models
from googleapiclient.errors import HttpError

import gspread
import google.auth
from IPython.display import display

# ── 1) Extrair apenas os DataFrames de destino ─────────────────────────────
if 'results' in locals() and results:
    dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}
    
    # ── 2) Validar consistência de datas ───────────────────────────────────────
    logger.info("🔍 Validando consistência de datas entre modelos …")
    df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

    if df_inconsistencies is not None and not df_inconsistencies.empty:
        logger.warning("💥 Inconsistências encontradas:")
        display(df_inconsistencies)
    else:
        logger.info("✅ Nenhuma divergência de start/end entre modelos.")
else:
    logger.warning("⚠️ Variável 'results' não existe ou está vazia. Pulando validação de datas.")

# ── 3) Limpar caches se necessário ─────────────────────────────────────────
# Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher`)
try:
    if 'fetcher' in locals() and 'SHEET_NAMES' in locals():
        fetcher.refresh(SHEET_NAMES)
        logger.info("🔄 Cache do fetcher limpo com sucesso")
except HttpError as e:
    if e.resp.status == 403:
        logger.warning("⚠️ Não foi possível limpar cache devido a erro de permissão (403)")
        logger.info("Cache será mantido até que as permissões sejam corrigidas")
    else:
        logger.error(f"❌ Erro ao limpar cache: {e}")
except Exception as e:
    logger.error(f"❌ Erro inesperado ao limpar cache: {e}")

# Limpa cache da parametrização BI em memória
try:
    BIParamLookup._df = None
    BIParamLookup._last_load = 0.0
    logger.info("🔄 Cache BIParamLookup limpo")
except Exception as e:
    logger.error(f"❌ Erro ao limpar cache BIParamLookup: {e}")

# ── 4) Estatísticas de uso das planilhas ───────────────────────────────────
try:
    creds, _ = google.auth.load_credentials_from_file(
        CREDS_PATH, scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
    )
    gc = gspread.authorize(creds)
    ss = gc.open_by_key(SPREADSHEET_ID)

    stats = []
    for ws in ss.worksheets():
        rows = ws.row_count
        cols = ws.col_count
        cells = rows * cols
        stats.append((cells, ws.title, rows, cols))

    stats.sort(reverse=True)  # maiores primeiro
    logger.info("📊 Top 10 abas que mais ocupam células:")
    for cells, title, rows, cols in stats[:10]:
        logger.info(f"  • {title}: {rows}×{cols} = {cells:,} células")
        
except HttpError as e:
    if e.resp.status == 403:
        logger.warning("⚠️ Não foi possível obter estatísticas da planilha devido a erro de permissão (403)")
    else:
        logger.error(f"❌ Erro HTTP ao obter estatísticas: {e}")
except Exception as e:
    logger.error(f"❌ Erro ao obter estatísticas da planilha: {e}")

In [ ]:
# 8
# %% [code]
from treat.treat_pipeline import BIParamLookup
from googleapiclient.errors import HttpError

# — Limpa cache de leituras em lote (SheetsFetcher já instanciado como `fetcher` em células anteriores)
try:
    if 'fetcher' in locals() and 'SHEET_NAMES' in locals():
        fetcher.refresh(SHEET_NAMES)
        logger.info("🔄 Cache do fetcher limpo com sucesso")
except HttpError as e:
    if e.resp.status == 403:
        logger.warning("⚠️ Não foi possível limpar cache devido a erro de permissão (403)")
    else:
        logger.error(f"❌ Erro ao limpar cache: {e}")
except Exception as e:
    logger.error(f"❌ Erro inesperado ao limpar cache: {e}")

# — Limpa cache da parametrização BI em memória
try:
    BIParamLookup._df = None
    BIParamLookup._last_load = 0.0
    logger.info("🔄 Cache BIParamLookup limpo")
except Exception as e:
    logger.error(f"❌ Erro ao limpar cache BIParamLookup: {e}")

In [11]:
# 9
# — Forçar recarregamento dos parâmetros BI em qualquer ponto do notebook —
from treat.bi_param_utils import BIParamLookup

# Zera o cache interno para que a próxima chamada a .df() refaça o carregamento
BIParamLookup._df = None
BIParamLookup._last_load = 0.0

In [ ]:
# 10 – Validar consistência de datas entre modelos

from treat.utils.validations import validate_consistent_dates_across_models
from IPython.display import display

# Extrai apenas os DataFrames de destino
dest_dfs = {sheet: info["dest"] for sheet, info in results.items()}

# Roda a validação
df_inconsistencies = validate_consistent_dates_across_models(dest_dfs)

# Exibe resultados
if df_inconsistencies is not None and not df_inconsistencies.empty:
    display(df_inconsistencies)
else:
    logger.info("✅ Nenhuma divergência de start/end entre modelos.")

In [ ]:
import gspread, google.auth
from pprint import pformat
from googleapiclient.errors import HttpError

try:
    creds, _ = google.auth.load_credentials_from_file(
        CREDS_PATH, scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
    )

    gc = gspread.authorize(creds)
    ss = gc.open_by_key(SPREADSHEET_ID)

    stats = []
    for ws in ss.worksheets():
        rows = ws.row_count
        cols = ws.col_count
        cells = rows * cols
        stats.append((cells, ws.title, rows, cols))

    stats.sort(reverse=True)  # maiores primeiro
    logger.info("📊 Top 40 abas que mais ocupam células:")
    logger.info(pformat(stats[:40]))  # top 40 abas que mais ocupam células
    
except HttpError as e:
    if e.resp.status == 403:
        logger.error("❌ Erro de permissão (403) ao obter estatísticas da planilha")
        logger.error("Certifique-se de que:")
        logger.error("  • A planilha está compartilhada com a conta de serviço")
        logger.error("  • A conta de serviço tem pelo menos permissão de Viewer")
    else:
        logger.error(f"❌ Erro HTTP {e.resp.status}: {e}")
except Exception as e:
    logger.error(f"❌ Erro ao obter estatísticas da planilha: {e}")